## Multipourpose Python scripts
First we need to find a way to retrieve all the K01220 from the KEGG database, we willl use the API from the database in order to do that.  


In [1]:
import requests
import re 
import pandas as pd

The `get_EC()` function retrieves a text response from a get petition given a $\text{Enzyme Commission Number}$ and prints an error if the response status code is other than 200

In [3]:
def get_EC(EC):
    URL = f'https://rest.kegg.jp/get/{EC}'
    response = requests.get(URL)
    if response.status_code == 200:
        data = response.text.splitlines()
        return data
    else:
        print(f"Error when trying to reach {URL}.  code: {response.status_code}")
        return none

data = get_EC('3.2.1.85')

The `parse_data()` function selects the desired information from `data` and stores it in lists, in this case `organisms` and `gene_list`, also it removes the `':'` character appended to the 3 letter code for each organism. 
The main idea is to iterate from these two list and from each element from the lists construct an URL and launch a get petition to retrive another necessary data.

In [142]:
def parse_data(data):
    organisms, gene_list = [], []
    field = ''
    for each_line in data:
        if each_line.startswith('GENES'):
            aux = each_line[12:].split()
            clean_name = aux[0].replace(':', '').lower()
            organisms.append(clean_name)
            gene_list.append(aux[1:])
            field = 'genes'
        elif each_line.startswith('            ') and field == 'genes':
            aux = each_line[12:].split()
            clean_name = aux[0].replace(':', '').lower()
            organisms.append(clean_name)
            gene_list.append(aux[1:])
        else:
            field = 'other'
    return organisms, gene_list
organisms, gene_list = parse_data(data)

But first we need to address a new issue. It turns out that for each organism exist at least one entry of the protein, this is a problem because this makes the future get petition a little more complex.  
The `match_orgs_genes(organisms, gene_list)` function takes the last created lists and creates two new lists `final_orgs` and `final_genes` from which increases the number of times an organism appears deppending on how many gene entries exist, i.e.  
If your lists looks like:  
  
organisms = ["OrgA", "OrgB"]  
gene_list = [["g1", "g2"], ["g3"]]  
  
The `match_orgs_genes()` function returns:  
final_orgs = ['OrgA', 'OrgA', 'OrgB']  
final_genes = ['g1', 'g2', 'g3']


In [146]:
def match_orgs_genes(organisms, gene_list):
    final_orgs ,final_genes = [], []
    for orgs, gene in zip(organisms, gene_list):
        for i in gene:
            final_orgs.append(orgs)
            final_genes.append(i)
    return final_orgs, final_genes
    
final_orgs, final_genes = match_orgs_genes(organisms, gene_list)

There is a little problem with the `final_genes` list, this is that some of the data with the gene appended to it, something like:  

final_genes = ['SMUGS5_06655', 'SMULJ23_0629(lacG)']  

So we want to separate the id entry from the gene, and the results must be appended into two new lists.  

id_entry = ['SMUGS5_06655', 'SMULJ23_0629']  
genes = [None, 'lacG']



In [148]:
def sep_gene_id(final_genes):
    id_entry, genes = [],[]
    for i in final_genes:
        gene = re.search(r'\([^)]*\)', i)
        if gene:
            id_ = re.sub(r'\([^)]*\)', '', i)
            id_entry.append(id_.strip())
            genes.append(gene.group().replace('(', '').replace(')', ''))
        else:
            id_entry.append(i.strip())
            genes.append(None)
    return id_entry, genes
id_entry, genes = sep_gene_id(final_genes)

Now, given a two lists each one containing a set of entrys and the three letter code we are going to iterate both of them in order to complete an URL and then start making get petitions 

In [ ]:
def waste_time(): #a function with the sole purpose to waste time and avoid any possible anti-bot function from the KEEG api
    time.sleep(random.uniform(2.8,6.0))
    
def get_seqs(test_orgs, test_gene):
    all_aa_length, all_aa_seq, all_nt_length, all_nt_seq = [], [], [], []
    
    for org, gene in zip(test_orgs, test_gene):
        URL = f'https://rest.kegg.jp/get/{org}:{gene}'
        aa_length, aa_seq, nt_length, nt_seq =[], [], [], []
    
        try:
            response = requests.get(URL, timeout=10)
            response.raise_for_status()
            data = response.text.splitlines()
            current_field = None
            
            for each_line in data:
                if each_line.startswith('AASEQ'):
                    aa_length.append(int(each_line[12:].strip()))
                    current_field = 'AASEQ'
                    continue
                if each_line.startswith('            ') and current_field == 'AASEQ':
                    aa_seq.append(each_line[12:].strip())
                    continue
                    
                if each_line.startswith('NTSEQ'):
                    nt_length.append(int(each_line[12:].strip()))
                    current_field = 'NTSEQ'
                    continue
                if each_line.startswith('            ') and current_field == 'NTSEQ':
                    nt_seq.append(each_line[12:].strip())
                    continue
                else:
                    current_field = None
                    
            aa_seq = ''.join(aa_seq)
            nt_seq = ''.join(nt_seq)
        
        except Exception as e:
            print(f"Error when trying to reach {URL}: {e}")
            aa_length.append(None)
            aa_seq.append(None)
            nt_length.append(None)
            nt_seq.append(None)
    
        all_aa_length.append(aa_length[0])
        all_aa_seq.append(aa_seq if isinstance(aa_seq, str) else aa_seq[0])
        all_nt_length.append(nt_length[0])
        all_nt_seq.append(nt_seq if isinstance(nt_seq, str) else nt_seq[0])
        
        waste_time()
    
    return all_aa_length, all_aa_seq, all_nt_length ,all_nt_seq

In [486]:
def chunkify(chunk_size, test_orgs, test_gene):
    if len(test_orgs) == len(test_gene):
        all_aa_length, all_aa_seq, all_nt_length ,all_nt_seq = [], [], [], []
        
        for i in range(0, len(test_orgs), chunk_size):
            chunk_orgs = test_orgs[i:i+chunk_size]
            chunk_gene = test_gene[i:i+chunk_size]

            aa_length, aa_seq, nt_length ,nt_seq =  get_seqs(chunk_orgs, chunk_gene)
            
            all_aa_length.extend(aa_length)
            all_aa_seq.extend(aa_seq)
            all_nt_length.extend(nt_length)
            all_nt_seq.extend(nt_seq)

            if i + chunk_size < len(test_orgs):
                time.sleep(random.uniform(200, 400))
            
        return all_aa_length, all_aa_seq, all_nt_length ,all_nt_seq
    else:
        print("ERROR: The lenght of given lists it's not the same, try again")

In [493]:
all_aa_length, all_aa_seq, all_nt_length ,all_nt_seq = chunkify(25, final_orgs, id_entry)

In [497]:
len(all_nt_seq)

409

In [460]:
test_gene

['SD1155_00675', 'PVA44_07260', 'PVA45_07505']

In [476]:
test_orgs=final_orgs[:5]
test_gene=id_entry[:5]

In [499]:
final_dic = {'Organisms':final_orgs, 'id_Entry':id_entry, 'Genes': genes, 'amino_Length':all_aa_length, 'amino_Sequence':all_aa_seq, 
             'nucl_Length':all_nt_length, 'nucl_Sequence':all_nt_seq}
final_df = pd.DataFrame(data=final_dic)

In [513]:
final_df

,Organims,id_Entry,Genes,amino_Length,amino_Sequence,nucl_Length,nucl_Sequence
0,sdo,SD1155_00675,lacG,470,MTKKLPEDFIFGGATAAYQAEGATQTDGKGRVAWDTYLEENYWYTA...,1413,atgactaaaaaattacctgaagattttatttttggtggagcgactg...
1,enem,PVA44_07260,lacG,477,MTKYTFPENFFFGAATAAYQAEGATDIGGKGKVAWDEMYHRESSTF...,1434,atgactaaatatacctttccagagaattttttctttggagcagcaa...
2,eet,PVA45_07505,lacG,477,MPKYTFTDNFFFGAATAAYQAEGATDVGGKGKVAWDEMYHRANSTF...,1434,atgcctaaatatacctttacagataacttcttttttggagcagcca...
3,bay,RBAM_012180,lacG,466,MLKLPHDFIFGGATAAYQAEGATKEGGKGPVAWDEFLEKQGRFSPD...,1401,atgctcaaattacctcatgactttattttcggcggggcgactgcgg...
4,baq,BACAU_1170,lacG,466,MLKLPHDFIFGGATAAYQAEGATKEGGKGPVAWDEFLEKQGRFSPD...,1401,atgctcaaattacctcatgactttattttcggcggggcgactgcgg...
...,...,...,...,...,...,...,...
404,lrug,AB8B22_00510,lacG,467,MKKLPEDFIFGGATAAYQAEGAIKIDGKGPVAWDKFLEENYWYTAE...,1404,atgaaaaaattaccagaagattttatttttggaggagcaacagcgg...
405,lmes,AB8B23_09530,lacG,467,MSKKLPEDFIFGGATAAYQAEGAIKTDGKGPVAWDKFLEENYWYTA...,1404,atgtcaaaaaaattaccagaagattttattttcggtggagcgacag...
406,lala,AB8B28_03175,lacG,467,MSKKLPEDFIFGGATAAYQAEGATKTDGKGRVAWDTFLEENYWYTA...,1404,atgtcaaaaaaattaccagaagattttatttttggaggagcaacag...
407,str,Sterm_3459,None,468,MERLPEDFIFGAATAAFQAEGAVNEDGRGKCYWDEYLHRAESTFNG...,1407,atggaaagattgcctgaagattttattttcggggctgcaactgcgg...


In [515]:
final_df.to_csv(r"C:\Users\jesus\Desktop\datos_seguro.txt", sep=",", index=False)